# Instrument Enrichment Debug (DataFrame Join)
Loads Supabase instruments + FinanceDatabase, performs normalized joins, previews updates, and optionally upserts.

In [ ]:
import os
import re
import json
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import financedatabase as fd

repo_root = Path('/Users/raphaelkim/Documents/Side Projects')

def load_env_file(path: Path):
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        v = v.strip()
        if k and k not in os.environ:
            os.environ[k] = v

def compact_text(value):
    text = '' if value is None else str(value)
    text = text.replace('\u00a0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    if text.lower() in {'', 'nan', 'none', 'null'}:
        return ''
    return text

def to_short_description(value, max_len=180):
    text = compact_text(value)
    if not text:
        return None
    if len(text) <= max_len:
        return text
    trimmed = text[: max_len - 1]
    cut = max(trimmed.rfind('. '), trimmed.rfind('; '), trimmed.rfind(', '), trimmed.rfind(' '))
    if cut >= int(max_len * 0.55):
        trimmed = trimmed[:cut]
    return trimmed.strip() + '...'

def normalize_symbol(raw):
    text = compact_text(raw).upper().lstrip('^')
    if ':' in text:
        text = text.split(':')[-1]
    if ' ' in text:
        text = text.split(' ', 1)[0]
    return text

def symbol_aliases(symbol):
    base = normalize_symbol(symbol)
    if not base:
        return []
    out = {base, base.replace('.', '-'), base.replace('-', '.'), base.replace('/', '-'), base.replace('/', '.')}
    for token in list(out):
        if '.' in token:
            out.add(token.split('.', 1)[0])
        if '-' in token:
            out.add(token.split('-', 1)[0])
    return [x for x in out if x]

def tags_list(*vals):
    out = []
    for val in vals:
        t = compact_text(val)
        if not t:
            continue
        if t.lower() not in [x.lower() for x in out]:
            out.append(t)
    return out

load_env_file(repo_root / '.env.local')
load_env_file(repo_root / '.env')
SUPABASE_URL = os.getenv('NEXT_PUBLIC_SUPABASE_URL')
SERVICE_KEY = os.getenv('SUPABASE_SERVICE_ROLE_KEY')
assert SUPABASE_URL and SERVICE_KEY, 'Missing Supabase env vars'
print('Supabase env loaded')

In [ ]:
# 1) Pull Supabase instruments into DataFrame
SCAN_LIMIT = int(os.getenv('INSTRUMENT_ENRICH_SCAN_LIMIT', '10000'))
endpoint = f'{SUPABASE_URL}/rest/v1/instruments'
query = urlencode({
    'select': 'symbol,name,asset_type,description,short_description,sector,industry,tags,is_active',
    'is_active': 'eq.true',
    'order': 'symbol.asc',
    'limit': str(SCAN_LIMIT)
}, safe='(),')
req = Request(endpoint + '?' + query, headers={'apikey': SERVICE_KEY, 'Authorization': f'Bearer {SERVICE_KEY}'})
with urlopen(req, timeout=120) as r:
    supa_rows = json.loads(r.read().decode('utf-8'))

df_supa = pd.DataFrame(supa_rows)
df_supa['symbol'] = df_supa['symbol'].map(normalize_symbol)
print('Supabase rows:', len(df_supa))
display(df_supa.head(10))

In [ ]:
# 2) Load FinanceDatabase and build normalized equity/ETF DataFrames
eq = fd.Equities().select()
etf = fd.ETFs().select()

df_eq = eq.reset_index()
if 'symbol' not in df_eq.columns:
    df_eq['symbol'] = df_eq.iloc[:, 0]
df_eq['symbol'] = df_eq['symbol'].map(normalize_symbol)
df_eq = df_eq[df_eq['symbol'] != '']

if 'summary' not in df_eq.columns:
    df_eq['summary'] = None
if 'sector' not in df_eq.columns:
    df_eq['sector'] = None
if 'industry' not in df_eq.columns:
    df_eq['industry'] = None

df_eq['short_description'] = df_eq['summary'].map(to_short_description)
df_eq['asset_type_enriched'] = 'equity'
df_eq['tags_enriched'] = df_eq.apply(lambda r: tags_list('equity', r.get('sector'), r.get('industry')), axis=1)
df_eq = df_eq[['symbol', 'asset_type_enriched', 'sector', 'industry', 'short_description', 'tags_enriched']].drop_duplicates('symbol')

df_etf = etf.reset_index()
if 'symbol' not in df_etf.columns:
    df_etf['symbol'] = df_etf.iloc[:, 0]
df_etf['symbol'] = df_etf['symbol'].map(normalize_symbol)
df_etf = df_etf[df_etf['symbol'] != '']

for c in ['category_group', 'category', 'family']:
    if c not in df_etf.columns:
        df_etf[c] = None

df_etf['short_description'] = df_etf.apply(lambda r: to_short_description(' '.join([
    f"ETF category: {compact_text(r.get('category_group'))}." if compact_text(r.get('category_group')) else '',
    f"Focus: {compact_text(r.get('category'))}." if compact_text(r.get('category')) else '',
    f"Provider: {compact_text(r.get('family'))}." if compact_text(r.get('family')) else ''
])), axis=1)
df_etf['sector'] = df_etf['category_group']
df_etf['industry'] = df_etf['category']
df_etf['asset_type_enriched'] = 'etf'
df_etf['tags_enriched'] = df_etf.apply(lambda r: tags_list('etf', r.get('category_group'), r.get('category'), r.get('family')), axis=1)
df_etf = df_etf[['symbol', 'asset_type_enriched', 'sector', 'industry', 'short_description', 'tags_enriched']].drop_duplicates('symbol')

print('Equity rows:', len(df_eq), 'ETF rows:', len(df_etf))
display(df_eq.head(5))
display(df_etf.head(5))

In [ ]:
# 3) Build alias-expanded mapping for joins
def expand_alias_df(df):
    records = []
    for _, r in df.iterrows():
        aliases = symbol_aliases(r['symbol'])
        for a in aliases:
            rec = r.to_dict()
            rec['alias_symbol'] = a
            records.append(rec)
    out = pd.DataFrame(records)
    out = out.drop_duplicates('alias_symbol')
    return out

eq_alias = expand_alias_df(df_eq)
etf_alias = expand_alias_df(df_etf)
print('eq alias rows:', len(eq_alias), 'etf alias rows:', len(etf_alias))

In [ ]:
# 4) Join Supabase -> ETF first, then fill from equity
base = df_supa.copy()
base['alias_symbol'] = base['symbol']

m1 = base.merge(etf_alias, on='alias_symbol', how='left', suffixes=('', '_etf'))
m2 = base.merge(eq_alias, on='alias_symbol', how='left', suffixes=('', '_eq'))

joined = base[['symbol','name','asset_type','description','short_description','sector','industry','tags']].copy()
joined['asset_type_new'] = m1['asset_type_enriched'].combine_first(m2['asset_type_enriched'])
joined['sector_new'] = m1['sector_etf'].combine_first(m2['sector_eq'])
joined['industry_new'] = m1['industry_etf'].combine_first(m2['industry_eq'])
joined['short_description_new'] = m1['short_description_etf'].combine_first(m2['short_description_eq'])
joined['tags_new'] = m1['tags_enriched_etf'].combine_first(m2['tags_enriched_eq'])

joined['matched'] = joined['short_description_new'].notna() | joined['sector_new'].notna() | joined['industry_new'].notna()
print('matched rows:', int(joined['matched'].sum()), 'of', len(joined))
display(joined[joined['matched']].head(20))
display(joined[~joined['matched']].head(20))

In [ ]:
# 5) Build update payload (only enrich fields; keep existing non-null values)
target = joined.copy()

target['asset_type_up'] = target['asset_type'].where(target['asset_type'].notna() & (target['asset_type'].astype(str).str.strip() != ''), target['asset_type_new'])
target['sector_up'] = target['sector'].where(target['sector'].notna() & (target['sector'].astype(str).str.strip() != ''), target['sector_new'])
target['industry_up'] = target['industry'].where(target['industry'].notna() & (target['industry'].astype(str).str.strip() != ''), target['industry_new'])
target['short_description_up'] = target['short_description'].where(target['short_description'].notna() & (target['short_description'].astype(str).str.strip() != ''), target['short_description_new'])
target['description_up'] = target['short_description_up']
target['tags_up'] = target['tags'].where(target['tags'].notna(), target['tags_new'])

for col in ['asset_type_up','sector_up','industry_up','short_description_up','description_up']:
    target[col] = target[col].replace({np.nan: None})

updates_df = target[target[['asset_type_up','sector_up','industry_up','short_description_up','tags_up']].notna().any(axis=1)][['symbol','asset_type_up','sector_up','industry_up','short_description_up','description_up','tags_up']].copy()
updates_df = updates_df.rename(columns={
    'asset_type_up':'asset_type',
    'sector_up':'sector',
    'industry_up':'industry',
    'short_description_up':'short_description',
    'description_up':'description',
    'tags_up':'tags'
})
print('potential updates:', len(updates_df))
display(updates_df.head(30))

## Optional: Upsert to Supabase
Uncomment and run only after reviewing `updates_df`.

In [ ]:
# upsert_rows = updates_df.to_dict(orient='records')
# endpoint = f"{SUPABASE_URL}/rest/v1/instruments?on_conflict=symbol"
# chunk_size = 500
# for i in range(0, len(upsert_rows), chunk_size):
#     chunk = upsert_rows[i:i+chunk_size]
#     req = Request(
#         endpoint,
#         method='POST',
#         data=json.dumps(chunk).encode('utf-8'),
#         headers={
#             'apikey': SERVICE_KEY,
#             'Authorization': f'Bearer {SERVICE_KEY}',
#             'Content-Type': 'application/json',
#             'Prefer': 'resolution=merge-duplicates'
#         }
#     )
#     with urlopen(req, timeout=120):
#         pass
#     print(f'Upserted {min(i+len(chunk), len(upsert_rows))}/{len(upsert_rows)}')
# print('Done')